In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

current_dir = Path.cwd()

if (current_dir / "data").exists():
    project_root = current_dir
elif (current_dir.parent / "data").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError("Cannot find project root.")

raw_path = (
    project_root
    / "data"
    / "raw"
    / "dst_reviews_english_5000.csv"
)

processed_path = (
    project_root
    / "data"
    / "processed"
    / "dst_reviews_cleaned.csv"
)

df = pd.read_csv(raw_path)

print("Raw shape:", df.shape)
df.head()

Raw shape: (5000, 21)


,recommendation_id,language,review,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,timestamp_created,timestamp_updated,playtime_forever_minutes,playtime_at_review_minutes,num_games_owned,author_num_reviews,created_at,updated_at,playtime_forever_hours,playtime_at_review_hours
0,232146949,english,Klei Entertainment was whippin' up actual shi in a kettle. Boiling shi in a kettle. He was whippin' up actual poo in...,False,0,0,0.5,0,False,False,False,1785965086,1785965086,265,265,110,29,2026-08-05 21:24:46+00:00,2026-08-05 21:24:46+00:00,4.416667,4.416667
1,232134193,english,So much content! When you get over the starting hurdle of surviving its so fun to explore and find out what everythi...,True,0,0,0.5,0,False,False,False,1785952590,1785952590,15951,15621,0,1,2026-08-05 17:56:30+00:00,2026-08-05 17:56:30+00:00,265.850000,260.350000
2,232131462,english,Pretty good game is a bit hard for new players so its best if youre starting out to look some stuff up or have a fri...,True,0,0,0.5,0,False,True,False,1785949854,1785949854,814,515,9,2,2026-08-05 17:10:54+00:00,2026-08-05 17:10:54+00:00,13.566667,8.583333
3,232125450,english,Мне понравилась игра очень веселая с друзьями особенно но печально что поиграли всего два раза но всем рекомендую,True,0,0,0.5,0,True,False,False,1785944379,1785944379,267,267,0,1,2026-08-05 15:39:39+00:00,2026-08-05 15:39:39+00:00,4.450000,4.450000
4,232114765,english,awesome game!,True,0,0,0.5,0,True,False,False,1785934292,1785934292,1793,1564,268,52,2026-08-05 12:51:32+00:00,2026-08-05 12:51:32+00:00,29.883333,26.066667


In [2]:
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_rate_percent"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

missing_summary

,missing_count,missing_rate_percent
review,25,0.5
recommendation_id,0,0.0
language,0,0.0
voted_up,0,0.0
votes_up,0,0.0
votes_funny,0,0.0
weighted_vote_score,0,0.0
comment_count,0,0.0
steam_purchase,0,0.0
received_for_free,0,0.0


In [3]:
clean_df = df.copy()

In [4]:
clean_df["review"] = clean_df["review"].fillna("")

clean_df["review"] = clean_df["review"].str.strip()

empty_review_count = clean_df["review"].eq("").sum()

print("Empty reviews before removal:", empty_review_count)

clean_df = clean_df[
    clean_df["review"].ne("")
].copy()

print("Shape after removing empty reviews:", clean_df.shape)

Empty reviews before removal: 26
Shape after removing empty reviews: (4974, 21)


In [5]:
before_deduplication = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["recommendation_id"],
    keep="first",
).copy()

removed_duplicates = before_deduplication - len(clean_df)

print("Removed duplicate IDs:", removed_duplicates)
print("Current shape:", clean_df.shape)

Removed duplicate IDs: 0
Current shape: (4974, 21)


In [6]:
clean_df["created_at"] = pd.to_datetime(
    clean_df["created_at"],
    utc=True,
    errors="coerce",
)

clean_df["updated_at"] = pd.to_datetime(
    clean_df["updated_at"],
    utc=True,
    errors="coerce",
)

print("Invalid created dates:", clean_df["created_at"].isna().sum())
print("Invalid updated dates:", clean_df["updated_at"].isna().sum())

print("Earliest review:", clean_df["created_at"].min())
print("Latest review:", clean_df["created_at"].max())

Invalid created dates: 0
Invalid updated dates: 0
Earliest review: 2025-11-25 14:26:05+00:00
Latest review: 2026-08-05 21:24:46+00:00


In [7]:
clean_df["review_date"] = clean_df["created_at"].dt.date
clean_df["review_year"] = clean_df["created_at"].dt.year
clean_df["review_month"] = (
    clean_df["created_at"]
    .dt.to_period("M")
    .astype(str)
)

/var/folders/s5/pnc17rj95bj164bbffk33lfw0000gq/T/ipykernel_6861/1073955677.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  clean_df["created_at"]


In [8]:
playtime_columns = [
    "playtime_forever_minutes",
    "playtime_at_review_minutes",
    "playtime_forever_hours",
    "playtime_at_review_hours",
]

for column in playtime_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce",
    )

In [9]:
for column in playtime_columns:
    clean_df.loc[
        clean_df[column] < 0,
        column,
    ] = pd.NA

In [10]:
clean_df[
    [
        "playtime_at_review_hours",
        "playtime_forever_hours",
    ]
].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

,playtime_at_review_hours,playtime_forever_hours
count,4974.000000,4974.000000
mean,142.346073,172.187793
std,595.212811,615.135510
min,0.083333,0.083333
50%,29.383333,46.825000
75%,115.129167,153.750000
90%,336.763333,406.941667
95%,590.039167,675.790833
99%,1485.375000,1703.736167
max,28341.550000,28347.250000


In [11]:
clean_df["review_length_chars"] = (
    clean_df["review"].str.len()
)

clean_df["review_word_count"] = (
    clean_df["review"]
    .str.split()
    .str.len()
)

clean_df["review_length_category"] = pd.cut(
    clean_df["review_word_count"],
    bins=[-1, 5, 20, 50, 100, float("inf")],
    labels=[
        "Very short",
        "Short",
        "Medium",
        "Long",
        "Very long",
    ],
)

In [12]:
clean_df[
    [
        "review",
        "review_word_count",
        "review_length_category",
    ]
].head()

,review,review_word_count,review_length_category
0,Klei Entertainment was whippin' up actual shi in a kettle. Boiling shi in a kettle. He was whippin' up actual poo in...,31,Medium
1,So much content! When you get over the starting hurdle of surviving its so fun to explore and find out what everythi...,23,Medium
2,Pretty good game is a bit hard for new players so its best if youre starting out to look some stuff up or have a fri...,28,Medium
3,Мне понравилась игра очень веселая с друзьями особенно но печально что поиграли всего два раза но всем рекомендую,18,Short
4,awesome game!,2,Very short


In [13]:
label_summary = (
    clean_df["voted_up"]
    .value_counts()
    .rename(index={
        True: "Recommended",
        False: "Not recommended",
    })
    .to_frame("count")
)

label_summary["percentage"] = (
    label_summary["count"] / len(clean_df) * 100
).round(2)

label_summary

,count,percentage
voted_up,,
Recommended,4512,90.71
Not recommended,462,9.29


In [14]:
positive_rate = clean_df["voted_up"].mean() * 100

print(f"Cleaned positive rate: {positive_rate:.2f}%")

Cleaned positive rate: 90.71%


In [15]:
processed_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

clean_df.to_csv(
    processed_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:", processed_path)
print("Final shape:", clean_df.shape)

Saved to: /Users/zhouzhou/Desktop/dst-review-analysis/data/processed/dst_reviews_cleaned.csv
Final shape: (4974, 27)


In [16]:
print("Rows:", len(clean_df))
print("Duplicate IDs:", clean_df["recommendation_id"].duplicated().sum())
print("Missing reviews:", clean_df["review"].isna().sum())
print("Empty reviews:", clean_df["review"].eq("").sum())
print("Positive reviews:", clean_df["voted_up"].sum())
print("Negative reviews:", (~clean_df["voted_up"]).sum())
print(
    "Positive rate:",
    round(clean_df["voted_up"].mean() * 100, 2),
)
print(
    "Date range:",
    clean_df["created_at"].min(),
    "to",
    clean_df["created_at"].max(),
)

Rows: 4974
Duplicate IDs: 0
Missing reviews: 0
Empty reviews: 0
Positive reviews: 4512
Negative reviews: 462
Positive rate: 90.71
Date range: 2025-11-25 14:26:05+00:00 to 2026-08-05 21:24:46+00:00


In [17]:
%pip install langdetect==1.0.9


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
from langdetect import DetectorFactory, LangDetectException, detect

DetectorFactory.seed = 0

In [19]:
def detect_review_language(text):
    """Detect review language.

    Very short reviews are marked separately because
    automatic language detection is unreliable for them.
    """

    if not isinstance(text, str):
        return "unknown"

    cleaned_text = " ".join(text.split())

    if len(cleaned_text) < 20:
        return "unknown_short"

    try:
        return detect(cleaned_text)
    except LangDetectException:
        return "unknown"

In [20]:
clean_df["detected_language"] = (
    clean_df["review"]
    .apply(detect_review_language)
)

In [21]:
language_summary = (
    clean_df["detected_language"]
    .value_counts()
    .to_frame("count")
)

language_summary["percentage"] = (
    language_summary["count"]
    / len(clean_df)
    * 100
).round(2)

language_summary.head(15)

,count,percentage
detected_language,,
en,2813,56.55
unknown_short,1760,35.38
ru,63,1.27
da,35,0.70
pt,32,0.64
no,24,0.48
af,20,0.40
es,20,0.40
id,15,0.30


In [22]:
non_english_sample = clean_df.loc[
    ~clean_df["detected_language"].isin(
        ["en", "unknown_short"]
    ),
    [
        "review",
        "detected_language",
        "voted_up",
    ],
]

non_english_sample.head(20)

,review,detected_language,voted_up
3,Мне понравилась игра очень веселая с друзьями особенно но печально что поиграли всего два раза но всем рекомендую,ru,True
14,В Don't Starve Together я играл несколько лет но в стиме купил недавно и хочу поделится о самой игре\nDon't Starve T...,ru,True
16,⠀⠀⢀⣀⠤⠿⢤⢖⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀\r\n⡔⢩⠂⠀⠒⠗⠈⠀⠉⠢⠄⣀⠠⠤⠄⠒⢖⡒⢒⠂⠤⢄⠀⠀⠀⠀\r\n⠇⠤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠀⠀⠈⠀⠈⠈⡨⢀⠡⡪⠢⡀⠀\r\n⠈⠒⠀⠤⠤⣄⡆⡂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠢⠀⢕⠱⠀...,unknown,True
25,Симулятор погоды в Питере,ru,True
33,"start game, die, start game, die, invite friends, start game, die but with laughs\n\n9/10 might do again",af,True
37,ggoddd survivallll <3,sv,True
39,Um dos melhores jogos ja criados. Sem dúvida,pt,True
60,Купил в раннем доступе,ru,True
68,i am sooo hungryyyyy mmm burgger pleeaseeeeee i want sburgerr ughhh yesssss,af,True
87,Very enjoyable game!,no,True


In [23]:
short_review_sample = clean_df.loc[
    clean_df["detected_language"] == "unknown_short",
    [
        "review",
        "voted_up",
        "review_word_count",
    ],
]

print("Short reviews:", len(short_review_sample))

short_review_sample.head(20)

Short reviews: 1760


,review,voted_up,review_word_count
4,awesome game!,True,2
7,Very Nice,True,2
9,fun if with friends,True,4
10,Good open world,True,3
11,i like it,True,3
13,fun game :>,True,3
15,great game pootis,True,3
22,very fun game,True,3
23,good,True,1
28,peak.,True,1


In [24]:
text_df = clean_df.loc[
    clean_df["detected_language"] == "en"
].copy()

print("All cleaned reviews:", len(clean_df))
print("English text reviews:", len(text_df))
print(
    "Removed from text analysis:",
    len(clean_df) - len(text_df),
)

All cleaned reviews: 4974
English text reviews: 2813
Removed from text analysis: 2161


In [25]:
comparison = pd.DataFrame(
    {
        "dataset": [
            "All cleaned reviews",
            "English text reviews",
        ],
        "reviews": [
            len(clean_df),
            len(text_df),
        ],
        "positive_rate": [
            clean_df["voted_up"].mean() * 100,
            text_df["voted_up"].mean() * 100,
        ],
    }
)

comparison["positive_rate"] = (
    comparison["positive_rate"].round(2)
)

comparison

,dataset,reviews,positive_rate
0,All cleaned reviews,4974,90.71
1,English text reviews,2813,88.70


In [26]:
all_cleaned_path = (
    project_root
    / "data"
    / "processed"
    / "dst_reviews_cleaned_all.csv"
)

english_text_path = (
    project_root
    / "data"
    / "processed"
    / "dst_reviews_english_text.csv"
)

clean_df.to_csv(
    all_cleaned_path,
    index=False,
    encoding="utf-8-sig",
)

text_df.to_csv(
    english_text_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", all_cleaned_path)
print("Saved:", english_text_path)

Saved: /Users/zhouzhou/Desktop/dst-review-analysis/data/processed/dst_reviews_cleaned_all.csv
Saved: /Users/zhouzhou/Desktop/dst-review-analysis/data/processed/dst_reviews_english_text.csv


In [27]:
print("All cleaned shape:", clean_df.shape)
print("English text shape:", text_df.shape)

print("\nDetected languages:")
print(
    clean_df["detected_language"]
    .value_counts()
    .head(10)
)

print(
    "\nAll-data positive rate:",
    round(clean_df["voted_up"].mean() * 100, 2),
)

print(
    "English-text positive rate:",
    round(text_df["voted_up"].mean() * 100, 2),
)

All cleaned shape: (4974, 28)
English text shape: (2813, 28)

Detected languages:
detected_language
en               2813
unknown_short    1760
ru                 63
da                 35
pt                 32
no                 24
af                 20
es                 20
id                 15
fr                 15
Name: count, dtype: int64

All-data positive rate: 90.71
English-text positive rate: 88.7


In [28]:
languages_to_check = [
    "unknown_short",
    "da",
    "no",
    "af",
    "id",
]

for language in languages_to_check:
    print(f"\nLanguage: {language}")

    display(
        clean_df.loc[
            clean_df["detected_language"] == language,
            [
                "review",
                "voted_up",
                "review_word_count",
            ],
        ].head(10)
    )


Language: unknown_short


,review,voted_up,review_word_count
4,awesome game!,True,2
7,Very Nice,True,2
9,fun if with friends,True,4
10,Good open world,True,3
11,i like it,True,3
13,fun game :>,True,3
15,great game pootis,True,3
22,very fun game,True,3
23,good,True,1
28,peak.,True,1



Language: da


,review,voted_up,review_word_count
166,i died and gave up 0/10,False,6
235,spider army to destroy my friends,True,6
481,10/10 experience super beginner friendly,True,5
656,"me like spider, me like friends 10/10",True,7
789,10/10 best game ever!,True,4
808,"i got killed by a frog, 10/10",True,7
874,Let's starve together XDDD,True,4
985,spider man spider man,True,4
1072,Very fun game i like very much good game,True,9
1530,hardest indie game iv ever played,True,6



Language: no


,review,voted_up,review_word_count
87,Very enjoyable game!,True,3
248,[strike]Don't[/strike] Starve Together,True,3
549,i starved... first mistake,True,4
1149,i'm not starving together,True,4
1201,i like\nvery difficult,True,4
1302,i like it but am new,True,6
1311,very fun like minecraft but more enjoyable maybe?,True,8
1688,chaos ;-; but i like!,True,5
2037,super fun kinda like minecraft,True,5
2275,I started starving so ate my friend,True,7



Language: af


,review,voted_up,review_word_count
33,"start game, die, start game, die, invite friends, start game, die but with laughs\n\n9/10 might do again",True,18
68,i am sooo hungryyyyy mmm burgger pleeaseeeeee i want sburgerr ughhh yesssss,True,12
121,How to survive:\nDon't die.,True,5
267,DLC being half or room rent,False,6
429,You'll get spanked and you'll like it,True,7
642,good and hard like a ♥♥♥♥,True,6
945,goood world building,True,3
1283,Cold? Died\r\nWarm? Died\r\nRain? Died\r\nFrogs? Died\r\nLight? Died\r\nA lot of food? Died\r\nNot enough food? Died...,True,22
1904,"I died to bees, it was peak",True,7
2374,wx ruin rushing go woooo,True,5



Language: id


,review,voted_up,review_word_count
88,"game kontol steam kontol, game kebuka terus ke exit, sudah coba semua cara sama aja tolol",False,16
380,kad igras sa 3 digitalca moze da bude challenging,True,9
774,uh... im kinda hungry folk,True,5
814,game gila tapi membuat candu,True,5
1059,"game bagus, pastikan habis bermain ini kalian tidur saja jangan melakukan hal lain, game penurun iki",True,16
1321,Game bagus yang selalu memberi hadiah tiap hari,True,8
1341,"Memiliki konsep unik, survival horror dengan fitur dan konsep yang luas mulai dari world building, item, mobs, hingg...",True,33
1359,salah satu game survival tersulit yg pernah gw coba,True,9
1401,murah meriah bisa mabar,True,4
2449,it makes me hungry :(,True,5


In [29]:
import re

In [30]:
NON_LATIN_PATTERN = re.compile(
    "["
    "\u0400-\u04FF"  # Cyrillic
    "\u4E00-\u9FFF"  # Chinese
    "\u3040-\u30FF"  # Japanese
    "\uAC00-\uD7AF"  # Korean
    "\u0600-\u06FF"  # Arabic
    "]"
)


def contains_non_latin_script(text):
    """Check for common non-Latin writing systems."""

    if not isinstance(text, str):
        return False

    return bool(NON_LATIN_PATTERN.search(text))

In [31]:
clean_df["contains_non_latin_script"] = (
    clean_df["review"]
    .apply(contains_non_latin_script)
)

In [32]:
print(
    clean_df["contains_non_latin_script"]
    .value_counts()
)

contains_non_latin_script
False    4826
True      148
Name: count, dtype: int64


In [33]:
clean_df.loc[
    clean_df["contains_non_latin_script"],
    [
        "review",
        "detected_language",
        "voted_up",
    ],
].head(20)

,review,detected_language,voted_up
3,Мне понравилась игра очень веселая с друзьями особенно но печально что поиграли всего два раза но всем рекомендую,ru,True
14,В Don't Starve Together я играл несколько лет но в стиме купил недавно и хочу поделится о самой игре\nDon't Starve T...,ru,True
25,Симулятор погоды в Питере,ru,True
58,Игра с атмосферой,unknown_short,True
60,Купил в раннем доступе,ru,True
110,Меня испугала голубая птица в море,ru,True
182,养孩子游戏什么的……,unknown_short,True
190,Классно.,unknown_short,True
206,Залупа,unknown_short,True
210,заебумба,unknown_short,True


In [34]:
english_mask = (
    clean_df["detected_language"].eq("en")
    |
    (
        clean_df["detected_language"].eq("unknown_short")
        & ~clean_df["contains_non_latin_script"]
    )
)

text_df = clean_df.loc[english_mask].copy()

In [35]:
print("All cleaned reviews:", len(clean_df))
print("English text reviews:", len(text_df))
print(
    "Excluded from text analysis:",
    len(clean_df) - len(text_df),
)

print(
    "All-data positive rate:",
    round(clean_df["voted_up"].mean() * 100, 2),
)

print(
    "English-text positive rate:",
    round(text_df["voted_up"].mean() * 100, 2),
)

All cleaned reviews: 4974
English text reviews: 4511
Excluded from text analysis: 463
All-data positive rate: 90.71
English-text positive rate: 90.73


In [36]:
excluded_df = clean_df.loc[
    ~english_mask,
    [
        "review",
        "detected_language",
        "review_word_count",
        "voted_up",
    ],
].copy()

print("Excluded reviews:", len(excluded_df))

excluded_df.head(30)

Excluded reviews: 463


,review,detected_language,review_word_count,voted_up
3,Мне понравилась игра очень веселая с друзьями особенно но печально что поиграли всего два раза но всем рекомендую,ru,18,True
14,В Don't Starve Together я играл несколько лет но в стиме купил недавно и хочу поделится о самой игре\nDon't Starve T...,ru,88,True
16,⠀⠀⢀⣀⠤⠿⢤⢖⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀\r\n⡔⢩⠂⠀⠒⠗⠈⠀⠉⠢⠄⣀⠠⠤⠄⠒⢖⡒⢒⠂⠤⢄⠀⠀⠀⠀\r\n⠇⠤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠀⠀⠈⠀⠈⠈⡨⢀⠡⡪⠢⡀⠀\r\n⠈⠒⠀⠤⠤⣄⡆⡂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠢⠀⢕⠱⠀...,unknown,33,True
25,Симулятор погоды в Питере,ru,4,True
33,"start game, die, start game, die, invite friends, start game, die but with laughs\n\n9/10 might do again",af,18,True
37,ggoddd survivallll <3,sv,3,True
39,Um dos melhores jogos ja criados. Sem dúvida,pt,8,True
58,Игра с атмосферой,unknown_short,3,True
60,Купил в раннем доступе,ru,4,True
68,i am sooo hungryyyyy mmm burgger pleeaseeeeee i want sburgerr ughhh yesssss,af,12,True


In [37]:
english_text_path = (
    project_root
    / "data"
    / "processed"
    / "dst_reviews_english_text.csv"
)

text_df.to_csv(
    english_text_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", english_text_path)
print("Final English text shape:", text_df.shape)

Saved: /Users/zhouzhou/Desktop/dst-review-analysis/data/processed/dst_reviews_english_text.csv
Final English text shape: (4511, 29)
